# Feature Engineering cho Dầu, Cửa hàng & Giao dịch (Oil, Store & Transaction Features)

Notebook này triển khai ba nhóm feature từ file `feature_oil_and_store.py`:

## Các Feature được tạo ra

### Oil Features (Đặc trưng Giá dầu)
| Feature | Mô tả |
|---|---|
| `oil_price` | Giá dầu WTI đã ffill missing values |
| `oil_price_lag_7` | Giá dầu 7 ngày trước |
| `oil_price_lag_14` | Giá dầu 14 ngày trước |
| `oil_price_rolling_mean_28` | Rolling mean 28 ngày (shift-1 safe) |
| `oil_price_change_pct` | Phần trăm thay đổi giá dầu so với tuần trước |

### Store Features (Đặc trưng Cửa hàng)
| Feature | Mô tả |
|---|---|
| `store_type_enc` | Label encode loại cửa hàng (fit on train only) |
| `city_freq` | Frequency encode thành phố |
| `state_freq` | Frequency encode bang/tỉnh |

### Transaction Features (Chưa triển khai trong file gốc)
| Feature | Mô tả |
|---|---|
| `transactions_lag_7` | Số giao dịch cùng ngày tuần trước |
| `transactions_rolling_mean_7` | Rolling mean giao dịch 7 ngày |

## Tổng quan Pipeline
```
Load Data → Oil Features (ffill + lag + rolling)
    → Store Encoding (label encode + frequency encode)
    → Transaction Features (TODO) → Save Output
```

## Nhận xét về file gốc

Sau khi đối chiếu file `feature_oil_and_store.py` với mô tả yêu cầu, phát hiện **3 vấn đề**:

### 1. Thiếu Transaction Features
Theo mô tả, cần có hai feature giao dịch: `transactions_lag_7` (số giao dịch cùng ngày tuần trước) và `transactions_rolling_mean_7` (rolling average 7 ngày). Tuy nhiên, file gốc **hoàn toàn không triển khai** nhóm feature này.

### 2. Lỗi cấu trúc — Nested Function
Hàm `encode_store_features` bị **thụt lề (indent) nhầm** vào bên trong hàm `create_oil_features`, trở thành một nested function. Câu lệnh `return oil` cũng nằm bên trong hàm con này. Đây rõ ràng là lỗi formatting, không phải thiết kế có chủ đích.

### 3. `create_oil_features` không bao giờ trả về giá trị
Do `return oil` bị mắc kẹt bên trong `encode_store_features`, hàm `create_oil_features` **không có return statement** ở top-level. Gọi hàm này sẽ luôn trả về `None`.

## 1. Import Thư viện

Sử dụng **pandas** cho thao tác dữ liệu và **LabelEncoder** từ scikit-learn để mã hóa `store_type`.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

## 2. Đọc Dữ liệu

Pipeline cần các bộ dữ liệu sau:
- **`train_cleaned.csv`** — dữ liệu train chính (cần `date`, `store_nbr`).
- **`test_cleaned.csv`** — dữ liệu test (cần cho store encoding fit on train only).
- **`oil.csv`** — giá dầu WTI hàng ngày.
- **`stores_cleaned.csv`** — thông tin cửa hàng (`store_type`, `city`, `state`).

In [ ]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train  = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_test   = pd.read_csv(rf'{BASE_PATH}\processed\test_cleaned.csv')
df_oil    = pd.read_csv(rf'{BASE_PATH}\raw\oil.csv')
df_stores = pd.read_csv(rf'{BASE_PATH}\processed\stores_cleaned.csv')

print(f'Train shape  : {df_train.shape}')
print(f'Test shape   : {df_test.shape}')
print(f'Oil shape    : {df_oil.shape}')
print(f'Stores shape : {df_stores.shape}')
df_oil.head(3)

---
## Phần 1 — Oil Features

Giá dầu WTI (`dcoilwtico`) ảnh hưởng trực tiếp đến nền kinh tế Ecuador (quốc gia xuất khẩu dầu). Ta tạo các feature lag và rolling từ giá dầu, sử dụng `ffill` để xử lý missing values (ngày không giao dịch).

Hàm `create_oil_features` đã được **sửa lỗi return** so với file gốc — `return oil` giờ nằm đúng vị trí ở cuối hàm.

In [ ]:
def create_oil_features(oil_df):
    oil = oil_df.copy()
    oil["date"] = pd.to_datetime(oil["date"])
    oil = oil.sort_values("date").rename(columns={"dcoilwtico": "oil_price"})

    # Safe missing handling (no future info)
    oil["oil_price"] = oil["oil_price"].ffill()

    # Lag features (historical only)
    oil["oil_price_lag_7"] = oil["oil_price"].shift(7)
    oil["oil_price_lag_14"] = oil["oil_price"].shift(14)

    # Rolling trend (shift first to avoid leakage)
    oil["oil_price_rolling_mean_28"] = (
        oil["oil_price"].shift(1).rolling(28, min_periods=7).mean()
    )

    # Weekly shock
    oil["oil_price_change_pct"] = (
        oil["oil_price"] / oil["oil_price_lag_7"] - 1
    )

    return oil


# Test the function
oil_featured = create_oil_features(df_oil)
print(f'Oil featured shape: {oil_featured.shape}')
print(f'Columns: {oil_featured.columns.tolist()}')
oil_featured.head(10)

---
## Phần 2 — Store Encoding

Mã hóa các đặc trưng cửa hàng:
- **Label encode `store_type`** — fit trên train, transform trên cả train và test để tránh data leakage.
- **Frequency encode `city` và `state`** — tính tần suất xuất hiện trên train, map sang cả hai tập.

Hàm `encode_store_features` đã được **tách ra khỏi** `create_oil_features` (sửa lỗi nested function trong file gốc).

In [ ]:
def encode_store_features(train, test):

    train, test = train.copy(), test.copy()

    # 1. Label encode store_type (fit on train only)
    le = LabelEncoder()
    train["store_type_enc"] = le.fit_transform(train["store_type"])
    test["store_type_enc"] = le.transform(test["store_type"])

    # 2. Frequency encode city/state
    for col in ["city", "state"]:
        freq = train[col].value_counts(normalize=True)
        train[f"{col}_freq"] = train[col].map(freq)
        test[f"{col}_freq"] = test[col].map(freq).fillna(0)

    # 3. cluster keep the same

    return train, test


# Merge store info into train/test before encoding
store_cols = ['store_nbr', 'store_type', 'city', 'state']
train_with_store = df_train.merge(df_stores[store_cols], on='store_nbr', how='left')
test_with_store  = df_test.merge(df_stores[store_cols], on='store_nbr', how='left')

train_encoded, test_encoded = encode_store_features(train_with_store, test_with_store)

print(f'Train encoded shape: {train_encoded.shape}')
print(f'Test encoded shape : {test_encoded.shape}')
print(f'\nNew columns: store_type_enc, city_freq, state_freq')
train_encoded[['store_nbr', 'store_type', 'store_type_enc', 'city_freq', 'state_freq']].drop_duplicates('store_nbr').head(10)

---
## Phần 3 — Transaction Features (Chưa triển khai trong file gốc)

Theo mô tả yêu cầu, số lượng giao dịch (transactions) mỗi ngày tại mỗi cửa hàng là proxy tốt cho lượng khách hàng. Cần tạo hai feature:

| Feature | Mô tả |
|---|---|
| `transactions_lag_7` | Số giao dịch cùng ngày tuần trước (same day last week) |
| `transactions_rolling_mean_7` | Rolling average số giao dịch 7 ngày gần nhất |

**Lưu ý**: Chỉ được dùng **lag transactions** để tránh data leakage — không dùng giá trị transactions của ngày hiện tại.

File gốc `feature_oil_and_store.py` **không triển khai** nhóm feature này. Cell bên dưới là stub/TODO để minh họa cách tính.

In [ ]:
# =============================================================
# TODO: Transaction Features — CHƯA CÓ TRONG FILE GỐC
# File feature_oil_and_store.py không triển khai phần này.
# Dưới đây là cách tính theo mô tả yêu cầu.
# =============================================================

# def create_transaction_features(df, transactions_df):
#     """
#     Tạo transaction lag features từ transactions.csv.
#     Chỉ dùng lag values để tránh data leakage.
#     """
#     df = df.copy()
#     txn = transactions_df.copy()
#     txn["date"] = pd.to_datetime(txn["date"])
#
#     # Merge transactions vào dataframe chính
#     df = df.merge(txn[["date", "store_nbr", "transactions"]], on=["date", "store_nbr"], how="left")
#
#     # Sort theo store + date trước khi tính lag
#     df = df.sort_values(["store_nbr", "date"]).reset_index(drop=True)
#     grouped = df.groupby("store_nbr")["transactions"]
#
#     # Lag 7 — same day last week
#     df["transactions_lag_7"] = grouped.shift(7)
#
#     # Rolling mean 7 — shift(1) to avoid leakage
#     df["transactions_rolling_mean_7"] = (
#         grouped.shift(1)
#         .transform(lambda x: x.rolling(7, min_periods=1).mean())
#     )
#
#     return df

print('Transaction features: TODO — chưa được triển khai trong file gốc.')

## Phần 4 — Danh sách tên Feature

Định nghĩa các hằng số tên feature để dùng trong các bước downstream (chọn feature, xác thực đầu ra).

In [ ]:
OIL_FEATURE_NAMES = [
    'oil_price',
    'oil_price_lag_7',
    'oil_price_lag_14',
    'oil_price_rolling_mean_28',
    'oil_price_change_pct',
]

STORE_FEATURE_NAMES = [
    'store_type_enc',
    'city_freq',
    'state_freq',
]

# TODO: Uncomment khi transaction features được triển khai
# TRANSACTION_FEATURE_NAMES = [
#     'transactions_lag_7',
#     'transactions_rolling_mean_7',
# ]

ALL_OIL_STORE_FEATURES = OIL_FEATURE_NAMES + STORE_FEATURE_NAMES

print(f'Oil features   : {len(OIL_FEATURE_NAMES)}')
print(f'Store features : {len(STORE_FEATURE_NAMES)}')
print(f'Total features : {len(ALL_OIL_STORE_FEATURES)}')
print(f'\nAll features: {ALL_OIL_STORE_FEATURES}')

## Phần 5 — Chạy Pipeline

Áp dụng toàn bộ pipeline:
1. Tạo oil features và merge vào train.
2. Encode store features cho cả train và test.
3. Hiển thị kết quả tổng hợp.

In [ ]:
# --- Step 1: Oil features ---
oil_featured = create_oil_features(df_oil)

# Merge oil features vào train
df_train['date'] = pd.to_datetime(df_train['date'])
df_result = df_train.merge(oil_featured[['date'] + OIL_FEATURE_NAMES], on='date', how='left')

print(f'After oil merge shape: {df_result.shape}')
print(f'Oil NaN counts:')
print(df_result[OIL_FEATURE_NAMES].isna().sum())

# --- Step 2: Store encoding ---
store_cols = ['store_nbr', 'store_type', 'city', 'state']
df_result = df_result.merge(df_stores[store_cols], on='store_nbr', how='left')

df_test_tmp = df_test.copy()
df_test_tmp = df_test_tmp.merge(df_stores[store_cols], on='store_nbr', how='left')

df_result, df_test_out = encode_store_features(df_result, df_test_tmp)

print(f'\nAfter store encoding shape: {df_result.shape}')
print(f'\n=== Feature Summary ===')
print(df_result[ALL_OIL_STORE_FEATURES].describe().T[['mean', 'min', 'max']])
print(f'\n=== Sample rows ===')
df_result[['date', 'store_nbr'] + ALL_OIL_STORE_FEATURES].head(10)

## Lưu Kết quả

Lưu dataframe đã được bổ sung oil và store features vào thư mục processed.

In [ ]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_oil_store_features.csv'
df_result.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'Shape   : {df_result.shape}')